# claude

> Claude Code's models through the Agent SDK. An agent with its own harness, not a completion endpoint.

Your tools travel in the system prompt as `<tool_call>` tags, never as MCP. That is not a fallback.
An enterprise-managed configuration forbids every dynamic MCP server, and the prompt is the one
channel no policy can close. `ClaudeChat.local` is `False`.

The conversation travels the other way: as a Claude Code session transcript that the turn resumes,
so the model reads real messages -- text, pictures, documents, and its own tool calls answered in
place -- rather than a conversation flattened into one prompt. Rishi's base install includes `llmsurgery`, which writes
those transcript records.

In [ ]:
#| default_exp claude

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, atexit, json, os, re, shutil, sys, time, weakref
import claude_agent_sdk
from llmsurgery import ant
from base64 import b64encode
from pathlib import Path
from fastcore.all import store_attr, patch, ifnone, listify
from fastcore.aio import run_sync, iter_sync
from rishi import core
from rishi.core import *

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec', 'ToolCall']

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

One turn is one `query()` on a live session, or one `query()` that resumes a filed transcript.
Claude Code has a real system-prompt channel, so the briefing goes there and the conversation
goes into the transcript.

The tools are the point. Claude Code declares a caller's tools to the model as an in-process MCP
server, and an organisation-managed configuration forbids every dynamic MCP server there is. On a
managed machine that path leaves the model with no tools at all. This backend never opens one. It
declares an empty MCP configuration, and the schemas go out as tags in the system prompt, which
`parse_tool_tags` reads back off the reply. A managed policy has nothing to refuse.

What it must *not* do is claim `strict_mcp_config=True`. That flag is refused outright where an
enterprise configuration exists ("You cannot use --strict-mcp-config when an enterprise MCP config
is present"). The obvious way to say "only my servers, please" is the one shape a managed machine
rejects. Declare nothing, claim nothing.

Two more options are contracts rather than tuning. `env` blanks `ANTHROPIC_API_KEY` for the
subprocess, because an ambient key silently turns a subscription session into a metered API one;
`api_key=True` leaves it alone for anyone who wants that. And `max_buffer_size` lifts the SDK's own
1MB stdout cap, which one long reply or one large tool result overruns -- the SDK raises rather than
truncating, so the turn is simply lost.

In [ ]:
#| export
CLAUDE_BIN = 'claude'   #: the binary the SDK spawns, overridden per chat with `bin=`
opus5    = 'claude-opus-5'
opus48   = 'claude-opus-4-8'
sonnet5  = 'claude-sonnet-5'
sonnet46 = 'claude-sonnet-4-6'
haiku45  = 'claude-haiku-4-5'
fable5   = 'claude-fable-5'

#: Every id above, for anything that wants to offer the list rather than reach for one of them.
CLAUDE_MODELS = {'opus5': opus5, 'opus48': opus48, 'sonnet5': sonnet5, 'sonnet46': sonnet46,
                 'haiku45': haiku45, 'fable5': fable5}
CLAUDE_DISALLOWED = ('Bash', 'Write', 'Edit', 'NotebookEdit')
CLAUDE_SERVER_TOOLS = ('WebSearch', 'WebFetch')  #: Claude Code's own, run on its side and never in rishi's tool loop
CLAUDE_WORK_DIR = Path.home()/'.rishi-claude'    #: where a conversation is filed, outside any real project
MAX_BUFFER = 10*2**20                            #: the SDK's stdout cap. Its own default is 1MB, and it raises rather than truncating

def claude_bin(bin=CLAUDE_BIN):
    "Absolute path to the `claude` binary, or a `FileNotFoundError` that says how to get one."
    if (p := shutil.which(bin)): return p
    raise FileNotFoundError(
        f'{bin!r} is not on $PATH. Install Claude Code (https://claude.com/claude-code) and run '
        f'`{bin} /login`. rishi drives it as a subprocess and never reads your credentials.')


def norm_claude_usage(u, model=None, cost=None):
    "Claude Code's usage block -> rishi's, so a Claude turn adds up with a local one."
    if not u: return {}
    read, made = u.get('cache_read_input_tokens') or 0, u.get('cache_creation_input_tokens') or 0
    pt, ct = (u.get('input_tokens') or 0) + read + made, u.get('output_tokens') or 0
    return {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct, 'cached_tokens': read,
            'cache_creation_tokens': made, 'cost': cost or 0.0, 'model': model}

ABORTED = ('aborted_streaming', 'aborted')
_status_re = re.compile(r'\b(\d{3})\b')

class ClaudeError(RuntimeError):
    "A failed Claude Code turn: what it said, the status behind it, and the raw result."
    def __init__(self, msg, status=None, raw=None):
        self.status, self.raw = status, raw
        super().__init__(msg)

def claude_status(d):
    "The HTTP status behind a failed result: what Claude Code reported, else one read out of the text."
    if (s := d.get('api_error_status')):
        try: return int(s)
        except (TypeError, ValueError): pass
    m = _status_re.search(f"{d.get('result') or ''} {d.get('subtype') or ''}")
    return int(m.group(1)) if m else None

def norm_claude(d, model=None):
    "A Claude Code result -> a rishi `Resp`, with `<tool_call>` tags read out of the text."
    if d.get('is_error') and d.get('terminal_reason') not in ABORTED:
        raise ClaudeError(f"claude failed: {d.get('result') or d.get('subtype')}",
                          status=claude_status(d), raw=d)
    text, th = split_think(d.get('result') or '')
    text, tcs = parse_tool_tags(text)
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    res['usage'] = norm_claude_usage(d.get('usage'), model, d.get('cost'))
    return Resp(res)
INTERRUPT_WAIT = 5
CONT_PROMPT = 'The tool results are in. Continue your answer.'


In [ ]:
test_fail(lambda: claude_bin('definitely-not-the-claude-binary'), contains='is not on $PATH')

r = norm_claude({'result': 'ok\n<tool_call>\n{"name": "ls", "arguments": {"path": "."}}\n</tool_call>',
                 'usage': {'input_tokens': 10, 'output_tokens': 5, 'cache_read_input_tokens': 90,
                           'cache_creation_input_tokens': 7}, 'cost': 0.0042}, opus5)
test_eq(resp_text(r), 'ok')
test_eq(r['tool_calls'][0]['function']['name'], 'ls')
test_eq(r['usage']['total_tokens'], 112)          # cache reads and writes are prompt tokens too
test_eq(r['usage']['cache_creation_tokens'], 7)
test_eq(r['usage']['cost'], 0.0042)               # what Claude Code itself billed, not a price table

# a failed turn carries the status, so a caller can tell a 429 from a bad prompt and retry on one
try:
    norm_claude({'is_error': True, 'result': 'API Error: 429 rate limited'}); assert False, 'no raise'
except ClaudeError as e:
    test_eq(e.status, 429)
    test_eq(e.raw['is_error'], True)
try:                                              # a reported status wins over one read out of the text
    norm_claude({'is_error': True, 'result': 'failed after 500 attempts', 'api_error_status': 529})
except ClaudeError as e: test_eq(e.status, 529)
try:                                              # ...and no status at all is not an error
    norm_claude({'is_error': True, 'subtype': 'error_during_execution'})
except ClaudeError as e: test_eq(e.status, None)
try:                                              # nor is one Claude Code spells with a word
    norm_claude({'is_error': True, 'result': 'nope', 'api_error_status': 'rate_limit'})
except ClaudeError as e: test_eq(e.status, None)
# `ClaudeError` is a `RuntimeError`, so anything already catching one still does
test_fail(lambda: norm_claude({'is_error': True, 'result': 'nope'}), contains='nope')
# an aborted turn is the user cancelling, not a failure: whatever arrived before is the reply
test_eq(resp_text(norm_claude({'is_error': True, 'terminal_reason': 'aborted', 'result': 'part'})), 'part')
for reason in ABORTED: norm_claude({'result': '', 'is_error': True, 'terminal_reason': reason})

## The chat

### One loop, and fastcore already owns it

`ClaudeSDKClient` holds an anyio task group open from `connect()` to `disconnect()`, so every call on it has to run on the loop it was built on. `fastcore.aio` keeps exactly one such loop on a daemon thread for the whole process: `run_sync` puts a coroutine on it and waits, `iter_sync` drives an async generator from sync code and closes it on the same loop. `sync_iter` in `rishi.core` builds a fresh loop per call, which is right for a one-shot `query()` and wrong for a session.

In [ ]:
#| export
def mk_claude_content(o):
    """`mk_oai_content`, plus the documents Claude Code takes and OpenAI content has no part for.

    A PDF rides in the same `image_url` envelope the other media use. That envelope is rishi's
    internal shape for "a data URL in the history"; the mime inside it is what `_to_parts` reads to
    choose between an Anthropic `image` block and a `document` one.
    """
    if isinstance(o, (dict, str)): return mk_oai_content(o)
    b = Path(o).read_bytes() if isinstance(o, os.PathLike) else o
    if isinstance(b, bytes):
        mime = core.detect_mime(b) or ''
        if not mime.startswith(('image/', 'audio/')):
            return {'type': 'image_url',
                    'image_url': {'url': f'data:{mime or "application/octet-stream"};base64,{b64encode(b).decode()}'}}
    return mk_oai_content(o)

def mk_claude_msg(content, role='user'):
    "`mk_oai_msg` built out of `mk_claude_content`."
    if content is None or isinstance(content, dict): return content
    parts = [mk_claude_content(o) for o in content] if isinstance(content, list) else [mk_claude_content(content)]
    if all(p.get('type') == 'text' for p in parts): return {'role': role, 'content': '\n'.join(p['text'] for p in parts)}
    return {'role': role, 'content': parts}

def mk_claude_msgs(msgs): return [mk_claude_msg(m) for m in listify(msgs)] if msgs else []


In [ ]:
#| export
class ClaudeChat(ToolLoopMixin, Chat):
    "Chat against a Claude Code model, with the same `rishi.core.Chat` API over the Agent SDK."
    _runtime = 'claude'
    _media_note = ("this Claude Code chat cannot carry a picture or a document: it is stateless and has "
                   "no transcript to file, so the conversation goes as one text prompt, which has nowhere "
                   "to put one. Keep the default `stateful=True`, leave `transcript=True`, or use "
                   "`rishi.remote`/`rishi.litert`.")
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_claude_content), staticmethod(mk_claude_msg), staticmethod(mk_claude_msgs)
    local, _ctx_live = False, 0

    def __init__(self, model=None, *, runtime=None, model_path=None, sp='', messages=None, tools=None,
                 ctx_limit=None, approve=None, tool_max_len=None, max_steps=10, parallel_tools=False,
                 max_parallel_tools=None, final_prompt=dflt_final_prompt_,
                 permission_mode='auto',    # Claude Code's gate on its *own* tools. Yours are rishi's
                 claude_tools=None,         # Claude Code's own tools, allowlisted. None -> none at all when this chat carries tools
                 claude_disallowed=CLAUDE_DISALLOWED,   # ...and the ones it may never use
                 claude_server_tools=(),    # its web tools, which run on its side and never reach rishi's loop
                 workspace=None,            # directory Claude Code works in. None -> `CLAUDE_WORK_DIR`. See `_cwd`
                 effort=None,               # 'low'/'medium'/'high'/'xhigh'/'max'. None -> the default
                 bare=True,                 # answer as a model: no CLAUDE.md, skills, plugins or hooks
                 stateful=True,             # keep one Claude Code session per chat. See `_connect`
                 transcript=True,           # file the conversation as records rather than prose. See `_file_sess`
                 api_key=False,             # let an ambient `ANTHROPIC_API_KEY` reach the subprocess. See `_opts`
                 max_buffer=MAX_BUFFER,     # the SDK's stdout cap, in bytes
                 bin=CLAUDE_BIN, timeout=600, settings=None, cbs=None, default_cbs=True, **opts):
        self.model_id = core.split_runtime(model)[1] or opus5
        self._set_tools(tools)
        store_attr('permission_mode,claude_tools,claude_disallowed,claude_server_tools,workspace,effort,'
                   'bare,bin,timeout,settings,api_key,max_buffer,opts')
        self.stateful = stateful
        self.transcript = bool(transcript)
        self.client = self._last_res = None
        self._stale = False             # a cancelled turn leaves the transcript and `hist` disagreeing
        self._ctx_live = 0              # occupancy the session last reported. See `token_count`
        self._sent = 0                  # how much of `hist` the live session has seen
        self.ctx_limit, self._ctx_tokens = ctx_limit, 0
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, parallel_tools=parallel_tools,
                    max_parallel_tools=max_parallel_tools, final_prompt=final_prompt, cbs=cbs,
                    default_cbs=default_cbs)

    @property
    def in_session_mode(self):
        "Does this chat drive one live session, rather than a fresh `query` per turn?"
        return self.stateful

    @property
    def _media_ok(self):
        "A live session and a filed transcript both carry content blocks. A text prompt cannot."
        return self.in_session_mode or self.transcript

    @property
    def _cwd(self):
        """The directory Claude Code runs in, which is also where a filed transcript lives.

        `workspace` when you named one, so a chat pointed at a project keeps its records with it.
        Otherwise `CLAUDE_WORK_DIR`, to keep synthesized transcripts out of a real project's
        history; pass `workspace='.'` to opt back into the current directory.
        """
        d = Path(self.workspace).expanduser() if self.workspace else CLAUDE_WORK_DIR
        return d.resolve()

    @property
    def tool_channel(self):
        "Where this chat's tool schemas travel. Always the prompt, to get past a managed MCP policy."
        return 'tags'

    @property
    def token_count(self):
        """What the live session last reported holding, else an estimate of the prompt it would send.

        Never asks the session here. A status bar reads this on the UI thread every repaint, and a
        round-trip to the subprocess from there waits on the same loop the turn is using: the CLI
        froze mid-turn. `_session_turn` refreshes it on the loop instead, once per turn.
        """
        return self._ctx_live or est_tokens(self._prompt()) + est_tokens(self._sp())

    def _sp(self):
        "The briefing plus the tool schemas, for the one channel a managed MCP policy cannot close."
        return tag_tools_sp(self.toolspecs, self.sp)

    def _prompt(self):
        "This turn's whole conversation as text. The briefing has a channel of its own."
        return render_prompt(self.hist)

    def _note_usage(self, r):
        "Remember what the turn cost. This is billing volume, not occupancy. See `token_count`."
        self._ctx_tokens = (r.get('usage') or {}).get('total_tokens') or self._ctx_tokens
        return r

    @property
    def in_session(self):
        "Is a Claude Code session live for this chat? Safe on a half-built one: `__del__` reaches here."
        return getattr(self, 'client', None) is not None

    def _recreate_conv(self):
        "Drop the session, so the next turn opens a fresh one and refiles `hist` into it."
        self._disconnect()

    def close(self):
        "End the session. A leaked chat would otherwise leak a `node` process; the loop is fastcore's."
        self._disconnect()

In [ ]:
#| hide
# a picture has a channel now: a live session takes content blocks, and a filed transcript records
# them. Only the flattened prompt has nowhere to put one, and it says so
c = ClaudeChat(stateful=False, transcript=False)
c.turn_msg = c.mk_msg([b'\x89PNG\r\n\x1a\n' + bytes(40), 'what is this?'])
test_fail(c._check_media, contains='no transcript to file')

# a live session takes blocks whether or not a transcript is filed, and a filed one records them
for kw in (dict(), dict(stateful=False), dict(transcript=False)):
    c = ClaudeChat(**kw)
    c.turn_msg = c.mk_msg([b'\x89PNG\r\n\x1a\n' + bytes(40), 'what is this?'])
    c._check_media()          # no chat connects, so there is nothing to close


## The options

`claude_agent_sdk.query` is one turn, asynchronous, yielding messages. The options carry the refusal
to open an MCP server, the blanked API key, the lifted buffer cap, and -- when a conversation has
been filed -- the id to resume.

`claude_server_tools` is the one place Claude Code's own tools are a feature rather than a hazard:
`WebSearch` and `WebFetch` run on its side, so they never reach rishi's tool loop and never need a
schema in the prompt. Everything else Claude Code can do stays off while this chat carries tools.

In [ ]:
#| export
@patch
def _opts(self:ClaudeChat, sp, resume=None, fork=False):
    "Agent SDK options for one turn, with no MCP server for a managed policy to refuse."
    sdk = claude_agent_sdk
    cwd = self._cwd
    cwd.mkdir(parents=True, exist_ok=True)
    srv, native = list(self.claude_server_tools or ()), (list(self.claude_tools) if self.claude_tools is not None else None)
    kw = dict(model=self.model_id, system_prompt=sp or None, cwd=str(cwd),
              permission_mode=self.permission_mode, settings=self.settings,
              mcp_servers={}, strict_mcp_config=False,        # see `The wire`
              include_partial_messages=True,                 # token deltas
              max_buffer_size=self.max_buffer,               # the SDK's own 1MB cap loses a long reply
              env={} if self.api_key else {'ANTHROPIC_API_KEY': ''},   # a subscription session, not a metered one
              disallowed_tools=list(self.claude_disallowed or ()))
    if native is not None: kw['allowed_tools'] = native + srv
    elif srv: kw['allowed_tools'] = srv
    if srv: kw['tools'] = sorted(set((native or []) + srv))   # available: exactly these
    elif native is None and self.toolspecs: kw['tools'] = []  # rishi's tools are the only channel
    if self.effort: kw['effort'] = self.effort
    if self.bare: kw.update(setting_sources=[], skills=[])    # no CLAUDE.md, plugins, hooks or skills
    if resume: kw.update(resume=resume, fork_session=fork)
    return sdk.ClaudeAgentOptions(**{**kw, **self.opts})

@patch
def _result(self:ClaudeChat, res, text):
    "A `ResultMessage` as the dict `norm_claude` reads, cost and error status included."
    return {'result': res.result or ''.join(text), 'usage': res.usage, 'is_error': res.is_error,
            'subtype': res.subtype, 'terminal_reason': getattr(res, 'terminal_reason', None),
            'cost': res.total_cost_usd,
            'api_error_status': getattr(res, 'api_error_status', None)}

@patch
def _sdk_events(self:ClaudeChat, prompt, sp, resume=None):
    "One `query` as `(kind, value)` pairs: `thought`, `text`, and one final `result` dict."
    sdk = claude_agent_sdk
    async def _agen():
        text, res = [], None
        ait = sdk.query(prompt=claude_prompt(prompt), options=self._opts(sp, resume=resume)).__aiter__()
        while True:
            try: m = await asyncio.wait_for(ait.__anext__(), self.timeout)
            except StopAsyncIteration: break
            except asyncio.TimeoutError:
                raise TimeoutError(f'Claude Code sent nothing for {self.timeout}s') from None
            if isinstance(m, sdk.AssistantMessage):
                for b in m.content:
                    if isinstance(b, sdk.ThinkingBlock) and b.thinking: yield 'thought', b.thinking
                    elif isinstance(b, sdk.TextBlock) and b.text: text.append(b.text); yield 'text', b.text
            elif isinstance(m, sdk.ResultMessage): res = m
        if res is None: raise RuntimeError('the Agent SDK ended without a result')
        yield 'result', self._result(res, text)
    return sync_iter(_agen, stop=self._cancel)

### Pictures and documents

Claude Code takes Anthropic content blocks, not OpenAI ones -- `image_url` comes back a 400. The conversion already exists in packages rishi depends on: `aidialog` normalises to `Part`s and `fastllm.anthropic.denorm_user` writes the blocks, `document` for a PDF as readily as `image` for a picture. What rishi supplies is the bridge from its own OpenAI-shaped history to those `Part`s.

A live session carries them, and so does a filed transcript. Only `transcript=False` cannot: it renders the conversation to one text prompt, which has nowhere to put a picture. Anthropic takes no audio on any path.

In [ ]:
#| export
@patch
def _to_parts(self:ClaudeChat, msgs):
    "rishi's OpenAI-shaped messages as aidialog `Part`s, media included."
    from aidialog.msg_parts import Text, InputImage, InputAudio, InputFile
    out = []
    for m in msgs:
        who = core._roles.get(m.get('role'), m.get('role', '?'))   # private, so not via the star import
        c = m.get('content')
        if isinstance(c, str):
            if c: out.append(Text(f'## {who}\n{c}'))
            continue
        for prt in (c or []):
            if not isinstance(prt, dict): continue
            t = prt.get('type')
            if t == 'text': out.append(Text(f'## {who}\n{prt.get("text","")}'))
            elif t == 'image_url':
                media = core._to_media_part(prt)
                out.append(InputFile(text=media.text, mime=media.mime) if media.mime == 'application/pdf' else media)
            elif t == 'input_audio': raise TypeError(
                'Claude Code takes no audio input. Use `rishi.remote`, `rishi.ollama` or `rishi.litert`.')
    return out

In [ ]:
#| hide
parts = ClaudeChat()._to_parts([{'role': 'user', 'content': 'hello'}])
test_eq(parts[0].text, '## User\nhello')

parts = ClaudeChat()._to_parts([{'role': 'user', 'content': [
    {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,aGVsbG8='}}]}])
test_eq((parts[0].__class__.__name__, parts[0].mime), ('InputImage', 'image/png'))

parts = ClaudeChat()._to_parts([{'role': 'user', 'content': [
    {'type': 'image_url', 'image_url': {'url': 'data:application/pdf;base64,AAA', 'text': 'paper'}}]}])
test_eq((parts[0].__class__.__name__, parts[0].mime, parts[0].text),
        ('InputFile', 'application/pdf', 'data:application/pdf;base64,AAA'))

test_fail(lambda: ClaudeChat()._to_parts([{'role': 'user', 'content': [
    {'type': 'input_audio', 'input_audio': {'data': '', 'format': 'wav'}}]}]), contains='no audio input')

## The conversation, as records rather than prose

A Claude Code session is a JSONL transcript under `~/.claude/projects/<cwd>/<id>.jsonl`, and
`resume` replays one. So the history channel is a file: `anth_msgs` turns rishi's OpenAI-shaped
`hist` into Anthropic messages, `llmsurgery.ant` writes them as records, and the turn resumes the
id. The model then reads real messages -- a picture as an `image` block, a PDF as a `document` one,
and a tool call answered by a `tool_result` block carrying its id -- where `render_prompt` could only
offer `## User` and `## Tool result (add)` prose.

Two details are contracts with Claude Code rather than choices. An assistant record's content must
be a *list* of blocks: 2.1.238 calls `.some()` on it looking for tool calls, and `mk_rec` leaves a
text-only assistant turn as a bare string, so a resume of one dies with `content.some is not a
function`. And a `tool_use` id is normalised to a `toolu_` one, deterministically, so the
`tool_result` that answers it still matches after a refiling.

The session id is a hash of the messages, so the same conversation files to the same transcript and
refiling it is idempotent. `fork_session=True` on connect keeps the live session's own turns out of
that file.

What this does not do is change a turn *inside* a live session. Claude Code takes user turns there,
so a tool result mid-turn still goes up as text; the real blocks are what the next refiling writes.
`stateful=True` is therefore the fast shape -- one subprocess for the whole chat -- and
`stateful=False` the faithful one, filing every turn as records at the cost of a process each.
`transcript=False` intentionally uses `_prompt` to send flattened text and cannot carry media.

In [ ]:
#| export
def anth_blocks(content):
    "rishi's OpenAI-shaped message content as Anthropic content blocks."
    if not content: return []
    if isinstance(content, str): return [{'type': 'text', 'text': content}]
    out = []
    for p in content:
        if not isinstance(p, dict): continue
        t = p.get('type')
        if t == 'text':
            if p.get('text'): out.append({'type': 'text', 'text': p['text']})
        elif t == 'image_url':
            url = (p.get('image_url') or {}).get('url', '')
            head, sep, data = url.partition(';base64,')
            if not sep:                        # a link, not bytes: Anthropic takes no URL source here
                out.append({'type': 'text', 'text': url})
                continue
            mime = head[5:] if head.startswith('data:') else 'application/octet-stream'
            out.append({'type': 'document' if mime == 'application/pdf' else 'image',
                        'source': {'type': 'base64', 'media_type': mime, 'data': data}})
        elif t == 'input_audio': raise TypeError(
            'Claude Code takes no audio input. Use `rishi.remote`, `rishi.ollama` or `rishi.litert`.')
    return out

def tu_id(tid, ant):
    "A tool call's id as Anthropic spells one, the same way every time it is filed."
    tid = tid or ''
    return tid if tid.startswith('toolu_') else 'toolu_' + ant.stable_uuid(f'rishi-tu:{tid}').replace('-', '')[:24]

def anth_msgs(hist, ant):
    "rishi's history as Anthropic messages, with tool calls answered in place as real blocks."
    out, called = [], set()
    for m in hist:
        role = m.get('role')
        if role == 'system': continue                  # the briefing has a channel of its own
        if role == 'tool':
            tid, txt = tu_id(m.get('tool_call_id'), ant), str(ifnone(m.get('content'), ''))
            # a `tool_result` answering no `tool_use` is a 400, and a hand-built history can hold one
            blocks = ([{'type': 'tool_result', 'tool_use_id': tid, 'content': txt}] if tid in called
                      else [{'type': 'text', 'text': f"Tool result ({m.get('name', '?')})\n{txt}"}])
            role = 'user'
        else:
            blocks = anth_blocks(m.get('content'))
            if role == 'assistant':
                tus = [{'type': 'tool_use', 'id': tu_id(tc.get('id'), ant), 'name': tc_name(tc),
                        'input': parse_args((tc.get('function') or {}).get('arguments'))}
                       for tc in (m.get('tool_calls') or [])]
                called |= {b['id'] for b in tus}
                blocks += tus
        if not blocks: continue                        # an empty turn is not a message
        role = 'assistant' if role == 'assistant' else 'user'
        if out and out[-1]['role'] == role: out[-1]['content'] += blocks   # a transcript reads these as one turn
        else: out.append({'role': role, 'content': blocks})
    return out

def claude_prompt(prompt):
    "A turn's prompt for `query`: a string as it is, content blocks as the streaming-input shape."
    if isinstance(prompt, str): return prompt
    async def _one():
        yield {'type': 'user', 'message': {'role': 'user', 'content': prompt},
               'parent_tool_use_id': None, 'session_id': 'default'}
    return _one()

In [ ]:
#| export
@patch
def _file_sess(self:ClaudeChat, hist=None):
    "File `hist` as a resumable Claude Code transcript, returning its session id, or None if empty."
    if not self.transcript: return None
    msgs = anth_msgs(ifnone(hist, self.hist), ant)
    if not msgs: return None
    cwd = self._cwd
    cwd.mkdir(parents=True, exist_ok=True)
    sid = ant.stable_uuid(f'rishi-claude:{ant.canon(msgs)}')
    recs = ant.msgs2recs(msgs, key=sid, cwd=cwd, model=self.model_id, entrypoint='sdk-py')
    for r in recs:
        c = r['message']['content']
        # An assistant record's content must be a list: Claude Code calls `.some()` on it, and
        # `mk_rec` leaves a text-only assistant turn as a bare string. See the section above.
        if r['type'] == 'assistant' and isinstance(c, str): r['message']['content'] = [{'type': 'text', 'text': c}]
    return ant.save_sess(recs, sid, cwd)

@patch
def _split_turn(self:ClaudeChat):
    """What to file and what to send: `(past, prompt)`.

    A `tool_use` record has to be answered by the `tool_result` that carries its id, so a turn that
    already holds the result files the whole conversation and asks the model to carry on. Anything
    else files the past and sends the last message as the turn.
    """
    if self.hist and self.hist[-1].get('role') == 'tool': return self.hist, CONT_PROMPT
    return self.hist[:-1], self._as_turn(self.hist[-1:])

@patch
def _as_turn(self:ClaudeChat, msgs):
    "`msgs` as one prompt: the text itself for a lone user turn, headed prose for several, blocks for media."
    if any(is_media(p) for m in msgs for p in (m.get('content') or []) if isinstance(p, dict)):
        from aidialog.msg_parts import Msg
        from fastllm.anthropic import denorm_user
        return denorm_user(Msg(role='user', content=self._to_parts(msgs)))['content']
    if len(msgs) == 1 and msgs[0].get('role') == 'user' and not msgs[0].get('tool_calls'):
        return resp_text(msgs[0])            # a real user turn, with no role heading to read past
    return render_prompt(msgs)

In [ ]:
#| hide
# the blocks, with no Claude behind them. The OpenAI `image_url` shape is what Claude Code answers
# a 400 to, so what matters is that it never leaves rishi
png = b'\x89PNG\r\n\x1a\n' + bytes(40)
c = ClaudeChat(); c.hist = [mk_claude_msg([png, 'what is this?'])]
blocks = c._as_turn(c.hist)
test_eq([b['type'] for b in blocks], ['image', 'text'])
test_eq(blocks[0]['source']['media_type'], 'image/png')

# a PDF is a `document` block -- the part `mk_oai_content` has no room for at all
pdf = b'%PDF-1.4\n' + b'0' * 40
c.hist = [mk_claude_msg([pdf, 'summarise this'])]
test_eq([b['type'] for b in c._as_turn(c.hist)], ['document', 'text'])

# text alone still goes as a prompt, not as blocks
c.hist = [{'role': 'user', 'content': 'hello'}]
test_eq(c._as_turn(c.hist), 'hello')

# and audio says why rather than failing somewhere downstream
c.hist = [{'role': 'user', 'content': [{'type': 'input_audio', 'input_audio': {'data': '', 'format': 'wav'}}]}]
test_fail(lambda: c._as_turn(c.hist), contains='no audio input')

# a live session and a filed transcript both have somewhere to put a picture; a flat prompt does not
test_eq(ClaudeChat()._media_ok, True)
test_eq(ClaudeChat(stateful=False)._media_ok, True)
test_eq(ClaudeChat(stateful=False, transcript=False)._media_ok, False)

In [ ]:
#| hide
# the conversion, with no Claude behind it. A tool call and its result have to come back as blocks
# that reference each other, or the model reads a call it never sees answered
_ant = ant
hist = [{'role': 'system', 'content': 'ignored: the briefing has its own channel'},
        {'role': 'user', 'content': 'what is 2+2?'},
        {'role': 'assistant', 'content': 'checking', 'tool_calls': [
            {'id': 'call_abc123', 'function': {'name': 'add', 'arguments': '{"a": 2, "b": 2}'}}]},
        {'role': 'tool', 'name': 'add', 'tool_call_id': 'call_abc123', 'content': '4'},
        {'role': 'assistant', 'content': 'it is 4'}]
ms = anth_msgs(hist, _ant)
test_eq([m['role'] for m in ms], ['user', 'assistant', 'user', 'assistant'])
tu = ms[1]['content'][1]
test_eq((tu['type'], tu['name'], tu['input']), ('tool_use', 'add', {'a': 2, 'b': 2}))
tr = ms[2]['content'][0]
test_eq((tr['type'], tr['content']), ('tool_result', '4'))
test_eq(tr['tool_use_id'], tu['id'])              # the pair matches, which is the whole point
test_eq(tu['id'].startswith('toolu_'), True)      # ...spelled the way Anthropic spells one
test_eq(anth_msgs(hist, _ant), ms)                # and the same history always files the same way

# consecutive same-role messages fold into one turn, and an empty one is not a message at all
test_eq(len(anth_msgs([{'role': 'user', 'content': 'a'}, {'role': 'user', 'content': 'b'}], _ant)), 1)
test_eq(anth_msgs([{'role': 'assistant', 'content': ''}], _ant), [])

# a result answering a call that is not in this history is prose, not a `tool_result` Anthropic 400s on
orphan = anth_msgs([{'role': 'user', 'content': 'hi'},
                    {'role': 'tool', 'name': 'add', 'tool_call_id': 'nope', 'content': '4'}], _ant)
test_eq([b['type'] for b in orphan[0]['content']], ['text', 'text'])
assert 'Tool result (add)' in orphan[0]['content'][1]['text']

# a picture becomes an `image` block and a PDF a `document` one, from the one envelope rishi carries
png = b'\x89PNG\r\n\x1a\n' + bytes(40)
blks = anth_blocks([core.mk_oai_content(png), {'type': 'text', 'text': 'what is this?'}])
test_eq([b['type'] for b in blks], ['image', 'text'])
test_eq(blks[0]['source']['media_type'], 'image/png')
test_eq(anth_blocks([{'type': 'image_url', 'image_url': {'url': 'data:application/pdf;base64,AAA'}}])[0]['type'], 'document')
test_fail(lambda: anth_blocks([{'type': 'input_audio', 'input_audio': {}}]), contains='no audio input')
# a link is not bytes, and Anthropic takes no URL source in a transcript: it travels as text
test_eq(anth_blocks([{'type': 'image_url', 'image_url': {'url': 'https://x/y.png'}}]),
        [{'type': 'text', 'text': 'https://x/y.png'}])

In [ ]:
#| hide
# ...and the records that reach disk, still with no Claude behind them
c = ClaudeChat(sonnet46, workspace=Path(CLAUDE_WORK_DIR)/'_test')
sid = c._file_sess(hist)
recs = [json.loads(l) for l in (_ant.sess_file(sid, c._cwd)).read_text().splitlines()]
test_eq(len(recs), 4)
test_eq([r['type'] for r in recs], ['user', 'assistant', 'user', 'assistant'])
# every assistant record's content is a list: Claude Code calls `.some()` on it and a string is fatal
test_eq([isinstance(r['message']['content'], list) for r in recs if r['type'] == 'assistant'], [True, True])
test_eq(c._file_sess(hist), sid)                  # the same conversation files to the same session
test_eq(c._file_sess([]), None)                   # ...and nothing files to nothing

# a lone user turn goes up as itself; several messages keep the role headings that tell them apart
test_eq(c._as_turn([{'role': 'user', 'content': 'hello'}]), 'hello')
assert '## Tool result (add)' in c._as_turn(hist[3:]), c._as_turn(hist[3:])

# a turn holding a tool result files the whole conversation, so the `tool_use` in it is answered by
# the `tool_result` that carries its id -- filing all but the last message would leave one dangling
c.hist = hist[1:4]
past, prompt = c._split_turn()
test_eq(len(past), 3)
test_eq(prompt, CONT_PROMPT)
test_eq([b['type'] for b in anth_msgs(past, _ant)[-1]['content']], ['tool_result'])
c.hist = hist[1:2]                              # ...and an ordinary turn still sends its last message
past, prompt = c._split_turn()
test_eq((past, prompt), ([], 'what is 2+2?'))

### The session, and how much of it has been sent

`ToolLoopMixin._send` steps the model several times in one turn, and `hist` grows between steps: the assistant reply, then a tool result for each call. The session already holds the reply -- Claude Code wrote it -- so only what rishi added since the last send goes up. `_sent` is that watermark, and it skips the assistant message by identity, because `_run_tools` and `_finish_turn` append the very object the step returned.

`_connect` starts the watermark at what the resumed transcript already holds, so the first turn sends its user message and nothing more. That is why `_events` connects *before* computing the delta.

In [ ]:
#| export
_live_sessions = weakref.WeakSet()
@atexit.register
def _close_sessions():
    "Close every live session, so no `claude` subprocess outlives the interpreter."
    for c in list(_live_sessions):
        try: c._disconnect()
        except Exception: pass

@patch
def _connect(self:ClaudeChat):
    """The live session, opened on first use. `_sp()` is fixed here: the SDK has no way to change it later.

    The past is filed and resumed rather than replayed as prose, so the session opens already holding
    the conversation as records. `fork_session` keeps its own turns out of that file.
    """
    if self.client is not None: return self.client
    past, _ = self._split_turn()
    sid = self._file_sess(past)
    client = claude_agent_sdk.ClaudeSDKClient(options=self._opts(self._sp(), resume=sid, fork=True))
    run_sync(asyncio.wait_for(client.connect(), self.timeout))
    self.client = client            # `_events` connects before computing the delta, so the watermark
    self._sent = len(past) if sid else 0             # can start at what the resumed session holds
    _live_sessions.add(self)        # tracked so `atexit` can close it: `__del__` is too late
    return client

@patch
def _disconnect(self:ClaudeChat):
    "End the session if there is one. The loop outlives it, ready for the next."
    c = getattr(self, 'client', None)
    self.client, self._sent, self._last_res, self._stale = None, 0, None, False
    if c is None: return
    _live_sessions.discard(self)
    if sys.is_finalizing(): return   # the loop thread is stopped; `run_sync` would never return
    try: run_sync(asyncio.wait_for(c.disconnect(), self.timeout))
    except Exception: pass          # a dead subprocess is already disconnected

@patch
def _delta(self:ClaudeChat):
    "The slice of `hist` the session has not been sent, as one prompt."
    if self._sent < len(self.hist) and self.hist[self._sent] is self._last_res:
        self._sent += 1             # the session wrote this one; sending it back would echo it
    new = self.hist[self._sent:]
    self._sent = len(self.hist)
    return self._as_turn(new)

In [ ]:
#| hide
# exit-time cleanup must not propagate: one bad session cannot strand the others or crash exit
class _Boom:
    def _disconnect(self): raise RuntimeError('already dead')
_live_sessions.add(b := _Boom())
_close_sessions()                      # must not raise
_live_sessions.discard(b)
test_eq(sys.is_finalizing(), False)    # `_disconnect` only skips the loop during finalisation


In [ ]:
#| hide
# the watermark, with no Claude behind it: what a second step in one turn would send
c = ClaudeChat()
c.hist = [{'role':'user','content':'what is 2+2?'}]
test_eq(c._delta(), 'what is 2+2?')                # a lone user turn goes up as itself
test_eq(c._sent, 1)

res = {'role':'assistant','content':'checking', 'tool_calls':[{'id':'1','function':{'name':'add','arguments':'{}'}}]}
c._last_res = res                                  # what `_model_step` just returned
c.hist.append(res)                                 # ...as `_run_tools` appends it
c.hist.append({'role':'tool','name':'add','content':'4'})
d = c._delta()
assert 'checking' not in d, d                      # the session wrote it; it does not go back up
assert '4' in d, d                                 # the tool result does
test_eq(c._sent, 3)
test_eq(c._delta(), '')                            # nothing new

c._recreate_conv()                                 # a fresh session refiles everything
test_eq((c._sent, c.in_session), (0, False))
c.close()

In [ ]:
#| hide
# a stalled session raises rather than hanging: `timeout` is what makes that so
import claude_agent_sdk as _sdk
from fastcore.test import test_fail

def _stalled(prompt=None, options=None):
    async def _g():
        await asyncio.sleep(60)
        yield None
    return _g()

_real, _sdk.query = _sdk.query, _stalled
try:
    c = ClaudeChat(); c.timeout = 0.2
    test_fail(lambda: list(c._sdk_events('hi', '')), contains='sent nothing for 0.2s')
finally: _sdk.query = _real

## The two steps `ToolLoopMixin` drives

In [ ]:
#| export
@patch
def cancel(self:ClaudeChat):
    "Stop the turn, and tell the session to stop generating rather than only stopping the reader."
    out = super(ClaudeChat, self).cancel()
    if self.in_session:
        try: run_sync(asyncio.wait_for(self.client.interrupt(), INTERRUPT_WAIT))
        except Exception: pass
        self._stale = True
    return out

@patch
def _ctx_usage(self:ClaudeChat):
    "What the live session says its window holds, or None."
    if not self.in_session: return None
    try: return run_sync(asyncio.wait_for(self.client.get_context_usage(), INTERRUPT_WAIT))
    except Exception: return None

In [ ]:
#| hide
# `token_count` must never wait on the session: a status bar reads it on the UI thread every
# repaint, and a round-trip from there waits on the loop the turn is already using
import claude_agent_sdk as _sdk

class _HangingClient:
    def __init__(self, options=None): pass
    async def connect(self, prompt=None): pass
    async def disconnect(self): pass
    async def get_context_usage(self):
        await asyncio.sleep(30)          # a status read that never returns
        return {'totalTokens': 1}

_real_client, _sdk.ClaudeSDKClient = _sdk.ClaudeSDKClient, _HangingClient
try:
    c = ClaudeChat()
    c.hist = [{'role': 'user', 'content': 'hello'}]
    c._connect()
    t0 = time.monotonic()
    n = c.token_count                    # returns the estimate, without asking the session
    assert time.monotonic() - t0 < 1, 'token_count waited on the session'
    test_eq(n > 0, True)
    c._ctx_live = 4242                   # once a turn has reported, that is what it answers
    test_eq(c.token_count, 4242)
    c.close()
finally: _sdk.ClaudeSDKClient = _real_client

In [ ]:
#| export
@patch
def _session_turn(self:ClaudeChat, prompt):
    "One turn on the live session, in `_sdk_events`' `(kind, value)` shape."
    sdk = claude_agent_sdk
    client = self._connect()
    async def _agen():
        text, res = [], None
        await client.query(claude_prompt(prompt))
        ait = client.receive_response().__aiter__()
        while True:
            if self._cancel.is_set(): break
            try: m = await asyncio.wait_for(ait.__anext__(), self.timeout)
            except StopAsyncIteration: break
            except asyncio.TimeoutError:
                raise TimeoutError(f'Claude Code sent nothing for {self.timeout}s') from None
            if isinstance(m, sdk.AssistantMessage):
                for b in m.content:
                    if isinstance(b, sdk.ThinkingBlock) and b.thinking: yield 'thought', b.thinking
                    elif isinstance(b, sdk.TextBlock) and b.text: text.append(b.text); yield 'text', b.text
            elif isinstance(m, sdk.ResultMessage): res = m
        if res is None: raise RuntimeError('the Claude Code session ended without a result')
        # already on the loop, so this is a local await rather than a cross-thread wait
        try: self._ctx_live = (await client.get_context_usage() or {}).get('totalTokens') or self._ctx_live
        except Exception: pass
        yield 'result', self._result(res, text)
    return iter_sync(_agen())

@patch
def _events(self:ClaudeChat):
    "This turn's events: the delta on a live session, else one query resuming the filed conversation."
    if not self.in_session_mode:
        if not self.transcript: return self._sdk_events(self._prompt(), self._sp())
        past, prompt = self._split_turn()             # the past as records, this turn as the prompt
        return self._sdk_events(prompt, self._sp(), resume=self._file_sess(past))
    if self._stale: self._disconnect()   # a cancelled turn: start again from what rishi holds
    self._connect()                      # before the delta, not after: connecting must not
    return self._session_events(self._delta() or CONT_PROMPT)   # come between computing it and sending it

@patch
def _session_events(self:ClaudeChat, prompt):
    "`_session_turn`, with one reconnect if the session died before it said anything."
    started = False
    try:
        for o in self._session_turn(prompt): started = True; yield o
    except Exception:
        if started: raise
        self._disconnect(); self._connect()   # a fresh session, refiled from `hist` by `_connect`
        for o in self._session_turn(self._delta() or CONT_PROMPT): yield o

In [ ]:
#| hide
# connecting must not come between computing the delta and sending it. It did, so the watermark
# was reset to 0 after the first turn and turn 2 re-sent the whole conversation -- feeding the
# model its own reply as if the user had said it. A fake client, so no subprocess is spawned.
import claude_agent_sdk as _sdk

class _FakeClient:
    def __init__(self, options=None): pass
    async def connect(self, prompt=None): pass
    async def disconnect(self): pass

_real_client, _sdk.ClaudeSDKClient = _sdk.ClaudeSDKClient, _FakeClient
try:
    c = ClaudeChat()
    c.hist = [{'role': 'user', 'content': 'one'}]
    g = c._events()                       # connects, then computes the delta
    test_eq(c._sent, 1)                   # ...and the delta survives the connect
    test_eq(c.in_session, True)
    g.close()
    c.hist += [{'role': 'assistant', 'content': 'two'}, {'role': 'user', 'content': 'three'}]
    g = c._events()
    test_eq(c._sent, 3)                   # only what is new goes up, never the whole history again
    g.close(); c.close()
finally: _sdk.ClaudeSDKClient = _real_client

In [ ]:
#| export
@patch
def _model_step(self:ClaudeChat, max_output_tokens=None):
    "One wire call."
    out = next(v for k, v in self._events() if k == 'result')
    self._last_res = res = self._note_usage(norm_claude(out, self.model_id))
    return res

@patch
def _stream_step(self:ClaudeChat, max_output_tokens=None):
    "The same turn, streamed. Thinking gets its own channel, and tag calls are never rendered as prose."
    split, out = StreamSplit(), None
    for kind, v in self._events():
        if kind == 'thought': yield {'channels': {'thought': v}}
        elif kind == 'text': yield from split.feed(v)
        else: out = v
    yield from split.finish()
    if out is None: raise RuntimeError('the turn ended without a result')
    self._step_res = self._last_res = self._note_usage(norm_claude(out, self.model_id))

@patch
def _oneshot(self:ClaudeChat, prompt, sp='', think=None, max_tokens=None):
    "Stateless one-shot text: no history to file and no session to keep."
    out = next(v for k, v in self._sdk_events(prompt, sp) if k == 'result')
    return resp_text(norm_claude(out, self.model_id))

## Tests

Nothing here starts a model. What is asserted is the options and the records, which is where the
enterprise contract lives and the part that fails silently if it regresses.

In [ ]:
def _fn(query: str) -> str:
    "Search the code."
    return ''

c = ClaudeChat('claude/claude-opus-5', sp='be brief', tools=[_fn], claude_disallowed=('Bash',))
c.hist = [{'role': 'user', 'content': 'what is 2 plus 2?'}]
test_eq(c.model_id, 'claude-opus-5')          # the `claude/` prefix is stripped, the id is not
test_eq(c.local, False)

o = c._opts(c._sp())
# The enterprise contract: no dynamic server is declared, and `strict_mcp_config` is *not* claimed.
# A managed machine refuses that flag outright, so asking for it is how this path used to fail.
test_eq(o.mcp_servers, {})
test_eq(o.strict_mcp_config, False)
test_eq(o.model, 'claude-opus-5')
test_eq(o.disallowed_tools, ['Bash'])
# rishi's tools are the only tool channel: Claude Code's own are off while this chat carries any
test_eq(o.tools, [])
test_eq(ClaudeChat()._opts('').tools, None)   # with none of rishi's, its own are left as they were

# ...so the schemas have to be somewhere, and the system prompt is where
test_eq('be brief' in o.system_prompt and '"_fn"' in o.system_prompt and '<tool_call>' in o.system_prompt, True)
test_eq('be brief' not in c._prompt(), True)  # and the briefing is not also in the conversation
test_eq(c.tool_channel, 'tags')

# an ambient API key never reaches the subprocess: it would silently meter a subscription session
test_eq(o.env, {'ANTHROPIC_API_KEY': ''})
test_eq(ClaudeChat(api_key=True)._opts('').env, {})
# the SDK's own 1MB stdout cap raises rather than truncating, losing a long reply outright
test_eq(o.max_buffer_size, MAX_BUFFER)
# the harness's own context is off by default: this is a model, not the user's IDE agent
test_eq((o.setting_sources, o.skills), ([], []))
test_eq(ClaudeChat(bare=False)._opts('').setting_sources, None)

# Claude Code's web tools are the one place its own tools are a feature, and they are opt-in
w = ClaudeChat(claude_server_tools=CLAUDE_SERVER_TOOLS)._opts('')
test_eq(w.tools, ['WebFetch', 'WebSearch'])
test_eq(w.allowed_tools, list(CLAUDE_SERVER_TOOLS))

# the conversation is resumed from a file, and the session's own turns stay out of that file
o2 = c._opts('', resume=c._file_sess([{'role': 'user', 'content': 'earlier'}]), fork=True)
test_eq(o2.fork_session, True)
assert o2.resume, 'no session to resume'
test_eq(ClaudeChat(transcript=False)._file_sess([{'role': 'user', 'content': 'x'}]), None)

# transcripts stay out of a real project unless you name one
test_eq(ClaudeChat()._cwd, CLAUDE_WORK_DIR.resolve())
test_eq(ClaudeChat(workspace='.')._cwd, Path('.').resolve())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()